# 02 · Cypher query — read the graph through the MCP server

Schema introspection, multi-hop reads, vertex/edge parsing, and pagination over the flights graph built by demo 01.

In [1]:
import warnings; warnings.filterwarnings("ignore")   # quiet 3rd-party import warnings
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))   # examples/demos
from _common import clients, console
print("helpers ready — no API key needed (the MCP servers are pure tools)")

helpers ready — no API key needed (the MCP servers are pure tools)


## Schema + a multi-hop read

In [2]:
async def query():
    async with clients.cypher_client("mcp_flights","flights") as cy:
        schema=clients.data(await cy.call_tool("get_agensgraph_schema",{}))
        console.kv("labels", {k:v["count"] for k,v in schema.items()})
        busiest=clients.data(await cy.call_tool("read_agensgraph_cypher",
            {"query":'MATCH (a:"Airport")-[:"ROUTE"]->() RETURN a.iata AS iata, a.city AS city, count(*) AS routes ORDER BY routes DESC LIMIT 5'}))
        console.table([(r["iata"],r["city"],r["routes"]) for r in busiest["rows"]], headers=["iata","city","out_routes"])
        v=clients.data(await cy.call_tool("read_agensgraph_cypher",{"query":'MATCH (a:"Airport" {iata:\'JFK\'}) RETURN a'}))
        console.kv("vertex JFK", v["rows"][0]["a"])
await query()

  labels                     {'Airport': 6072}
  iata  city     out_routes
  ----  -------  ----------
  ATL   Atlanta  915       
  ORD   Chicago  558       
  LHR   London   527       
  PEK   Beijing  525       
  CDG   Paris    524       
  vertex JFK                 {'id': '3.2983', 'label': 'Airport', 'properties': {'lat': 40.63980103, 'lon': -73.77890015, 'city': 'New York', 'iata': 'JFK', 'name': 'John F Kennedy International Airport', 'country': 'United States'}}


## Pagination — bounded pages following `next_offset`

In [3]:
async def paginate():
    async with clients.cypher_client("mcp_flights","flights") as cy:
        q='MATCH (a:"Airport")-[r:"ROUTE"]->(b:"Airport") RETURN a.iata AS src, b.iata AS dst, r.airline AS airline'
        offset=0
        for n in range(3):
            page=clients.data(await cy.call_tool("read_agensgraph_cypher",{"query":q,"limit":1000,"offset":offset}))
            console.kv(f"page {n} (offset {offset})", f"{page['row_count']} rows, has_more={page['has_more']}, next={page['next_offset']}")
            offset=page["next_offset"]
        total=clients.data(await cy.call_tool("read_agensgraph_cypher",{"query":'MATCH ()-[r:"ROUTE"]->() RETURN count(*) AS n'}))
        console.kv("total routes", f"{total['rows'][0]['n']:,}")
await paginate()

  page 0 (offset 0)          1000 rows, has_more=True, next=1000


  page 1 (offset 1000)       1000 rows, has_more=True, next=2000
  page 2 (offset 2000)       1000 rows, has_more=True, next=3000
  total routes               66,934
